# OFAC Sanctions Screening with Briefcase AI - Interactive Walkthrough

## Overview
This notebook demonstrates how to use Briefcase AI to create compliant audit trails for **OFAC sanctions screening** in real-time payment processing.

### What You'll Learn:
- How to capture real-time sanctions screening decisions
- OFAC/BSA compliance requirements for watchlist version tracking
- How to maintain audit trails for blocked transactions
- Best practices for sanctions reporting timeline compliance

### Regulatory Context:
- **Regulation**: OFAC/BSA (Office of Foreign Assets Control)
- **Regulator**: FinCEN/Treasury
- **Requirements**: Real-time screening, watchlist provenance, reporting deadlines

## Step 1: Setup and Imports

In [ ]:
import sys
import os
import uuid
import random
import hashlib
from datetime import datetime, timedelta
from typing import Dict, Any

# Add shared module to path
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'shared'))

try:
    import backend
    from backend import briefcase, DecisionSnapshot, Input, Output, SqliteBackend
    print("[SUCCESS] Successfully imported Briefcase AI SDK")
except ImportError as e:
    print(f"[FAILED] Error importing required modules: {e}")
    print("Please ensure the Briefcase AI SDK is installed")

## Step 2: Initialize Briefcase AI and Generate Watchlist Version

OFAC compliance requires tracking the exact version of the sanctions watchlist used for each decision.

In [ ]:
# Initialize Briefcase AI SDK
try:
    briefcase.init_with_config(2)
    print("[SUCCESS] Briefcase AI SDK initialized")
except Exception as e:
    print(f"[FAILED] Failed to initialize SDK: {e}")

# Get configured backend
db_backend = backend.get_backend()
print("[SUCCESS] SQLite backend configured")

# Generate current watchlist version (simulated)
# In production, this would come from your actual OFAC watchlist system
current_watchlist_sha = hashlib.sha256(
    f"ofac-watchlist-{datetime.utcnow().strftime('%Y%m%d')}".encode()
).hexdigest()[:16]

print(f"\n[PROTECTED] Current OFAC Watchlist Version: {current_watchlist_sha}")
print(f"📅 Watchlist Date: {datetime.utcnow().strftime('%Y-%m-%d')}")
print(f"\n**Insight:** Watchlist version tracking is critical for OFAC compliance")

## Step 3: Simulate Payment Transaction Data

Create a wire transfer that will be screened against OFAC sanctions lists.

In [ ]:
# Generate a wire transfer for sanctions screening
transaction_id = str(uuid.uuid4())
payment_data = {
    "transaction_id": transaction_id,
    "originator_name": "Aleksei Petrov",  # This will trigger sanctions match
    "originator_country": "RU",
    "beneficiary_name": "John Smith",
    "payment_amount": 50000.0,
    "payment_type": "wire",
    "watchlist_version_sha": current_watchlist_sha,
    "screening_config_version": "ofac-screening-v2.3.1"
}

print("**Cost:** Payment Transaction for Screening:")
for key, value in payment_data.items():
    if key != "watchlist_version_sha":  # Don't clutter output with full SHA
        print(f"  • {key}: {value}")
print(f"  • watchlist_version: {current_watchlist_sha[:12]}...")

print(f"\n[WARNING] Risk Factors:")
print(f"  • Russian originator name: {payment_data['originator_name']}")
print(f"  • High value transaction: ${payment_data['payment_amount']:,.2f}")
print(f"  • Cross-border wire transfer")

## Step 4: OFAC Sanctions Screening Model

Simulate the AI model that performs real-time sanctions screening against OFAC watchlists.

In [ ]:
def simulate_ofac_screening_model(payment_data: Dict[str, Any]) -> Dict[str, Any]:
    """
    Simulates OFAC sanctions screening AI model.
    In production, this would query actual OFAC watchlists and use
    sophisticated name matching algorithms.
    """
    originator = payment_data["originator_name"].lower()
    country = payment_data["originator_country"]
    amount = payment_data["payment_amount"]

    # Simulate sanctions name matching
    sanctions_score = 0.0
    matched_entity = None
    
    # Check for high-risk names (simulated)
    high_risk_names = ["aleksei petrov", "vladimir", "dmitri"]
    for risk_name in high_risk_names:
        if risk_name in originator:
            sanctions_score = 0.95  # Very high match
            matched_entity = risk_name.title()
            break

    # Country risk factor
    high_risk_countries = ["RU", "IR", "KP", "BY"]
    if country in high_risk_countries:
        sanctions_score += 0.2
    
    # Amount threshold
    if amount > 10000:
        sanctions_score += 0.1

    # Ensure score doesn't exceed 1.0
    sanctions_score = min(sanctions_score, 1.0)

    # Decision thresholds
    decision_threshold = 0.7
    
    if sanctions_score >= decision_threshold:
        decision = "block"
        risk_level = "high"
    elif sanctions_score >= 0.3:
        decision = "review"
        risk_level = "medium"
    else:
        decision = "approve"
        risk_level = "low"

    return {
        "decision": decision,
        "sanctions_score": round(sanctions_score, 3),
        "matched_entity": matched_entity,
        "risk_level": risk_level,
        "decision_threshold": decision_threshold,
        "screening_timestamp": datetime.utcnow().isoformat(),
        "model_version": "ofac-screening-v4.1.2"
    }

# Run OFAC screening
print("[PROTECTED] Running OFAC sanctions screening...")
screening_result = simulate_ofac_screening_model(payment_data)

print(f"\n**Results:** Screening Results:")
print(f"  • Decision: {screening_result['decision'].upper()}")
print(f"  • Sanctions Score: {screening_result['sanctions_score']}")
print(f"  • Decision Threshold: {screening_result['decision_threshold']}")
print(f"  • Risk Level: {screening_result['risk_level']}")

if screening_result['matched_entity']:
    print(f"  • [WARNING] Matched Entity: {screening_result['matched_entity']}")
    print(f"  • 🚫 This transaction MUST be blocked per OFAC requirements")

## Step 5: Create OFAC Compliance Audit Trail

Capture the screening decision with all required OFAC compliance metadata.

In [ ]:
# Prepare OFAC/BSA regulatory metadata
regulatory_metadata = {
    "regulation": "OFAC/BSA",
    "watchlist_version_sha": current_watchlist_sha,
    "screening_timestamp": screening_result["screening_timestamp"],
    "requires_sar_filing": screening_result["decision"] == "block",
    "transaction_blocked": screening_result["decision"] == "block",
    "examiner_ready": True
}

# Add reporting deadline for blocked transactions
if screening_result["decision"] == "block":
    reporting_deadline = datetime.utcnow() + timedelta(days=30)  # 30-day OFAC reporting requirement
    regulatory_metadata["reporting_deadline"] = reporting_deadline.isoformat()

print("**Details:** OFAC Compliance Metadata:")
for key, value in regulatory_metadata.items():
    if key == "watchlist_version_sha":
        print(f"  • {key}: {str(value)[:12]}...")
    else:
        print(f"  • {key}: {value}")

# Create decision snapshot
try:
    decision_snapshot = backend.create_decision_snapshot(
        function_name="ofac_sanctions_screening",
        inputs=payment_data,
        outputs=screening_result,
        metadata=regulatory_metadata,
        input_types={
            "payment_amount": "float",
            "originator_name": "string",
            "beneficiary_name": "string"
        },
        output_types={
            "sanctions_score": "float",
            "decision_threshold": "float"
        }
    )
    print("\n[SUCCESS] OFAC decision snapshot created")
    
except Exception as e:
    print(f"\n[FAILED] Error creating decision snapshot: {e}")

## Step 6: Store in Immutable Audit Trail

In [ ]:
# Store decision in backend
try:
    stored_decision_id = db_backend.save_decision(decision_snapshot)
    print(f"[SUCCESS] OFAC decision stored in audit trail")
    print(f"[SECURED] Decision ID: {stored_decision_id}")
    
    if screening_result["decision"] == "block":
        print(f"\n🚫 TRANSACTION BLOCKED - Immediate Actions Required:")
        print(f"  • Customer notification: Transaction declined")
        print(f"  • SAR filing deadline: {regulatory_metadata['reporting_deadline'][:10]}")
        print(f"  • Funds frozen pending investigation")
    
except Exception as e:
    print(f"[FAILED] Error storing decision: {e}")

## Step 7: Retrieve and Display Audit Trail

In [ ]:
print("**Analysis:** AUDIT TRAIL DEMONSTRATION")
print("=" * 60)

# Load decision back from backend
try:
    retrieved_decision = db_backend.load_decision(stored_decision_id)
    if retrieved_decision:
        backend.print_audit_summary(retrieved_decision)
    else:
        print("[FAILED] Failed to retrieve decision from backend")
        
except Exception as e:
    print(f"[FAILED] Error retrieving decision: {e}")

## Step 8: FinCEN Examiner Simulation

Simulate how a FinCEN examiner would query OFAC compliance records.

In [ ]:
print("🏛 FINCEN EXAMINER SIMULATION")
print("=" * 60)

examiner_query = f"Show me the OFAC screening details for transaction {transaction_id} involving {payment_data['originator_name']}"
print(f"**Analysis:** EXAMINER QUERY: {examiner_query}")

examiner_response = backend.format_examiner_response(
    stored_decision_id,
    examiner_query,
    db_backend
)
print(examiner_response)

## Step 9: OFAC-Specific Compliance Validation

In [ ]:
print("⚖ OFAC-SPECIFIC VALIDATION")
print("=" * 60)

# Verify critical watchlist provenance
retrieved_watchlist_sha = retrieved_decision.tags.get("watchlist_version_sha")
if retrieved_watchlist_sha == current_watchlist_sha:
    print(f"[SUCCESS] Watchlist version integrity confirmed")
    print(f"  **Details:** Original SHA: {current_watchlist_sha}")
    print(f"  **Details:** Retrieved SHA: {retrieved_watchlist_sha}")
else:
    print(f"[FAILED] Watchlist version mismatch - CRITICAL COMPLIANCE ISSUE")
    print(f"  Expected: {current_watchlist_sha}")
    print(f"  Retrieved: {retrieved_watchlist_sha}")

# Check reporting deadline compliance for blocked transactions
if screening_result['decision'] == 'block':
    deadline = retrieved_decision.tags.get("reporting_deadline")
    if deadline:
        deadline_date = datetime.fromisoformat(deadline.replace('Z', '+00:00') if deadline.endswith('Z') else deadline)
        days_remaining = (deadline_date - datetime.utcnow()).days
        print(f"\n📅 Blocked transaction reporting deadline: {deadline_date.strftime('%Y-%m-%d')}")
        print(f"⏰ Days remaining for OFAC filing: {days_remaining}")
        
        if days_remaining >= 0:
            print(f"[SUCCESS] Reporting timeline compliant")
        else:
            print(f"[FAILED] Reporting deadline exceeded - CRITICAL VIOLATION")
    else:
        print("[FAILED] Missing reporting deadline for blocked transaction")

# Overall OFAC compliance validation
required_fields = [
    "regulation",
    "watchlist_version_sha", 
    "screening_timestamp",
    "transaction_blocked"
]

validation_result = backend.validate_regulatory_completeness(
    retrieved_decision,
    required_fields
)

status_icon = "[SUCCESS]" if validation_result['is_compliant'] else "[FAILED]"
status_text = "COMPLIANT" if validation_result['is_compliant'] else "NON-COMPLIANT"

print(f"\n{status_icon} OFAC Compliance Status: {status_text}")
print(f"**Results:** Completeness Score: {validation_result['completeness_score']:.1%}")

if validation_result['missing_fields']:
    print(f"[FAILED] Missing Required Fields: {', '.join(validation_result['missing_fields'])}")

## Summary

### What We Accomplished
[SUCCESS] **Created a complete OFAC/BSA compliant audit trail** for sanctions screening

[SUCCESS] **Captured all critical elements:**
- Transaction details and screening inputs
- Sanctions matching scores and decisions
- Exact watchlist version used
- Reporting deadlines for blocked transactions

[SUCCESS] **Demonstrated regulatory readiness:**
- FinCEN examiner query simulation
- Watchlist version integrity verification
- Compliance timeline tracking

### Key OFAC Compliance Benefits
- **Watchlist Provenance**: Exact version tracking prevents compliance gaps
- **Real-time Decisions**: Immediate blocking of sanctioned transactions
- **Reporting Timelines**: Automated deadline tracking for SAR filings
- **Audit Readiness**: Complete transaction history for examiner reviews

### Critical Compliance Points
🚫 **Blocked Transaction Actions:**
1. Immediately freeze transaction
2. Notify customer (generic decline message)
3. File SAR within 30 days
4. Maintain complete audit trail

**Details:** **Documentation Requirements:**
- Exact watchlist version used
- Screening algorithm and thresholds
- Decision rationale and timing
- Follow-up actions taken

**Transaction ID**: `{transaction_id}`  
**Decision ID**: `{stored_decision_id}`